# 《九曜：天墟》EP01 Reference 生成 — Google Colab 版

用途：在免費 Colab GPU（T4）上生成 EP01 的 3 個場景 + 5 個角色 Reference 圖，速度比本機 GTX 1660 Ti 快很多。

**使用方式**：
1. 選單「執行階段」→「變更執行階段類型」→ 硬體加速器選 **T4 GPU**，儲存
2. 從上到下依序執行每一格（Shift+Enter），或直接「全部執行」
3. 全部跑完後，最後一格會把 8 張圖打包成 `EP01_references.zip` 並自動下載
4. 下載後把 zip 解壓，貼回本機專案的 `production/EP01/references/` 底下對應資料夾（scenes / characters）

**如果 prompt 改過要重新生成**：請先選單「執行階段」→「中斷連線並刪除執行階段」，重新連線後再「全部執行」，確保抓到 GitHub 上最新版的腳本（Step 3 已經加了自動清除舊 clone 的保護，但保險起見還是建議整個重開執行階段）。

In [ ]:
# Step 1：確認拿到 GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Step 2：安裝依賴（Colab 已內建 torch+CUDA，只需要補 diffusers 這幾個）
!pip install -q diffusers transformers accelerate safetensors sentencepiece

In [ ]:
# Step 3：拉取專案 repo，取用 docs/30 規格與生成腳本的同一份 prompt
# 每次都先刪掉舊的 clone、清掉 import 快取，避免重跑時抓到舊版程式碼
!rm -rf /content/jiuyao-tianxu
!git clone --depth 1 https://github.com/sky03104/jiuyao-tianxu.git /content/jiuyao-tianxu
%cd /content/jiuyao-tianxu/production/EP01

import sys
for _mod in list(sys.modules):
    if _mod == 'generate_references':
        del sys.modules[_mod]

In [ ]:
# Step 4：T4 有 16GB VRAM，遠超 SDXL 需求，不用 cpu_offload，直接整包塞 GPU 跑更快
# 這裡覆寫本機腳本的 build_pipeline，改成全 GPU 常駐模式
import importlib, sys
sys.path.insert(0, '/content/jiuyao-tianxu/production/EP01')
import generate_references as gr
import torch
from diffusers import StableDiffusionXLPipeline

def build_pipeline_gpu(model_path: str):
    pipe = StableDiffusionXLPipeline.from_pretrained(
        model_path, dtype=torch.float16, use_safetensors=True
    )
    pipe = pipe.to('cuda')
    pipe.vae.enable_slicing()
    return pipe

gr.build_pipeline = build_pipeline_gpu
print('已切換為 Colab GPU 常駐模式')

In [ ]:
# Step 5：依文件順序生成 —— 先 3 個場景，再 5 個角色
# 換成 RealVisXL_V4.0：對「亂生文字/裝飾邊框」控制比 SDXL Base 好很多，寫實人物風格也更貼近需求
pipe = gr.build_pipeline('SG161222/RealVisXL_V4.0')

scene_keys = ['academy', 'corridor', 'room07']
char_keys = ['player', 'qi_henglie', 'yu_cenye', 'li_ruofeng', 'xiao_yaolin']

for k in scene_keys + char_keys:
    gr.generate_one(pipe, k)

In [ ]:
# Step 6：打包下載
import shutil
from google.colab import files

shutil.make_archive('/content/EP01_references', 'zip', '/content/jiuyao-tianxu/production/EP01/references')
files.download('/content/EP01_references.zip')